# Validation of fires A and B (WP 3.3)

Browses the outputs of `scripts/validate.py`: for every validation in the output directory, its
labels and warnings, the runs that reached the grid edge, the metrics table and the figures, then
any Rothermel/Alexandridis comparison. The method and the protocol are in
[docs/validation.md](../docs/validation.md).

This notebook computes nothing and is committed without outputs. **No real-fire results exist
yet**: they need the built regions and a calibration of fire A (docs/validation.md, section
*Results*). Until then the directory is empty and the notebook says what to run. Outputs of the
script's `--synthetic` mode are tests of the pipeline and say so in every table and figure.

The directory is `docs/img/validation/`, or the one in the environment variable
`VALIDATION_OUT` (for outputs written elsewhere, e.g. on a cluster).

In [ ]:
import csv
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = Path(os.environ.get("VALIDATION_OUT", ROOT / "docs" / "img" / "validation"))
summaries = sorted(OUT.glob("*_summary.json"))
comparisons = sorted(OUT.glob("*_rule_comparison.csv"))
if not summaries:
    print(f"No validation outputs in {OUT}. Once the regions are built and fire A is calibrated, run")
    print("  uv run --extra fast --extra geo python scripts/validate.py lake_2024 --workers 8")
    print("  uv run --extra fast --extra geo python scripts/validate.py silverado_2020 --workers 8")
    print("and the other commands of docs/validation.md, section Results.")

In [ ]:
def show_table(path):
    """Print a CSV file as a Markdown table."""
    with path.open(encoding="utf-8", newline="") as fh:
        rows = list(csv.reader(fh))
    print("| " + " | ".join(rows[0]) + " |")
    print("|" + " --- |" * len(rows[0]))
    for row in rows[1:]:
        print("| " + " | ".join(row) + " |")


def show_figure(path):
    """Display a PNG at its aspect ratio."""
    image = plt.imread(path)
    fig, ax = plt.subplots(figsize=(10, 10 * image.shape[0] / image.shape[1]))
    ax.imshow(image)
    ax.set_axis_off()

## Validations

One block per summary file: labels, parameters, edge count, metrics, figures.

In [ ]:
for path in summaries:
    summary = json.loads(path.read_text(encoding="utf-8"))
    prefix = path.name.removesuffix("_summary.json")
    params, edge = summary["parameters"], summary["edge"]
    print("=" * 100)
    print(f"{prefix}: {summary['label']} - {summary['target']['description']}")
    for warning in summary["warnings"]:
        print(f"WARNING: {warning}")
    print(f"parameters: {params['status']}, rule {params['rule']}, {params['params']} ({params['path']})")
    print(f"runs reaching the grid edge: {edge['n_reached']} of {edge['n_runs']} (milestone M2 requires 0)")
    for note in summary["target"]["notes"]:
        print(f"note: {note}")
    show_table(OUT / f"{prefix}_metrics.csv")
    for name in summary["files"]:
        if name.endswith(".png"):
            show_figure(OUT / name)

## Rothermel against Alexandridis

Tables written by `scripts/validate.py --compare`.

In [ ]:
for path in comparisons:
    print("=" * 100)
    print(path.name)
    show_table(path)